In [1]:
!pip install -q langchain langchain-community langchain-core langchain-groq
!pip install -q transformers accelerate bitsandbytes

Access is denied.
Access is denied.


In [ ]:
import os
from getpass import getpass

# Nhập Key
os.environ["GROQ_API_KEY"] = getpass("Nhập GROQ_API_KEY của bạn: ")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0,
            groq_api_key=os.getenv("GROQ_API_KEY")
        )

In [ ]:
from langchain.tools import tool
@tool
def subtract(a: float, b: float) -> float:
    """
    Trừ số 'b' khỏi số 'a'. Trả về kết quả của a - b.
    Ví dụ: subtract(10, 3) sẽ trả về 7.0.
    """
    return a - b

@tool
def divide(a: float, b: float) -> float:
    """
    Chia số 'a' cho số 'b'. Trả về kết quả của a / b (float).
    Xử lý trường hợp chia cho 0.
    Ví dụ: divide(10, 2) sẽ trả về 5.0.
    """
    if b == 0:
        return "Lỗi: Không thể chia cho 0."
    return a / b # Trả về float

tools = [subtract, divide]
from langchain.prompts import ChatPromptTemplate
from langchain.agents import create_react_agent, AgentExecutor
prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là một trợ lý tính toán giỏi. Bạn có quyền truy cập vào các công cụ để thực hiện phép trừ và phép chia."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])
agent = create_react_agent(llm, tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Câu hỏi cần thực thi
user_query = "Tính kết quả của (20 chia 4) trừ đi 2"

# Kết quả cuối cùng
result = agent_executor.invoke({"input": user_query})
print(f"\nKết quả cuối cùng: {result['output']}")

In [11]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Trả về tổng của a và b"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Trả về tích của a và b"""
    return a * b

tools = [add, multiply]

In [12]:
from langchain.prompts import ChatPromptTemplate
from langchain.agents import create_react_agent, AgentExecutor

prompt = ChatPromptTemplate.from_template(
    """Bạn là một trợ lý toán học thông minh có thể sử dụng các công cụ sau để tính toán:

{tools}

Tên các công cụ: {tool_names}

Hướng dẫn:
- Khi người dùng hỏi, hãy suy nghĩ từng bước.
- Chọn công cụ phù hợp (add hoặc multiply), thực hiện phép toán, rồi trả về kết quả cuối cùng.

Lịch sử suy nghĩ của bạn:
{agent_scratchpad}

Câu hỏi: {input}
"""
)

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
#  Tính 5 + 6
agent_executor.invoke({"input": "Tính 5 + 6"})

In [ ]:
#  Tính 5 * 6
agent_executor.invoke({"input": "Tính 5 * 6"})

In [ ]:
#  Thử thách: (2 + 3) * 4
agent_executor.invoke({"input": "Tính (2 + 3) * 4"})